In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
import joblib       # to save the trained model to disk.

In [6]:
df = pd.read_csv("../data/AAPL_features.csv", index_col="Price", parse_dates=True)
print(df.shape)
df.head()

(2467, 21)


,Close,High,Low,Open,Volume,return,lag1,lag2,lag3,sma5,...,sma50,price_to_sma20,price_to_sma50,volatility_10,volatility_20,rsi,volume_change,volume_ma20,volume_ratio,target
Price,,,,,,,,,,,,,,,,,,,,,
2015-03-16,27.758121,27.758121,27.296043,27.520417,143497200.0,0.011004,-0.006911,0.018079,-0.018232,27.535528,...,26.537694,0.978381,1.045988,0.012886,0.013712,32.710634,-0.307811,234379600.0,0.612243,1
2015-03-17,28.222429,28.284632,27.913636,27.969174,204092400.0,0.016727,0.011004,-0.006911,0.018079,27.647937,...,26.618291,0.995054,1.060265,0.014304,0.014193,45.533450,0.422274,231953740.0,0.879884,1
2015-03-18,28.540108,28.693394,28.073585,28.213541,261083600.0,0.011256,0.016727,0.011004,-0.006911,27.924741,...,26.718871,1.006353,1.068163,0.014748,0.014339,44.971637,0.279242,236029580.0,1.106148,0
2015-03-19,28.324615,28.713385,28.302400,28.602308,183238000.0,-0.007551,0.011256,0.016727,0.011004,28.060254,...,26.815098,0.999126,1.056294,0.013868,0.014433,47.391182,-0.298164,237719000.0,0.770818,0
2015-03-20,27.969170,28.524553,27.804776,28.491231,274780400.0,-0.012549,-0.007551,0.011256,0.016727,28.162889,...,26.897622,0.987982,1.039838,0.014503,0.014538,41.765681,0.499582,241668340.0,1.137014,1


In [7]:
features = [
    "lag1", "lag2", "lag3",
    "sma5", "sma20", "sma50",
    "price_to_sma20", "price_to_sma50",
    "volatility_10", "volatility_20",
    "rsi",
    "volume_change", "volume_ratio"
]

X = df[features]
y = df["target"]

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (2467, 13)
Target distribution:
target
1    1307
0    1160
Name: count, dtype: int64


#### 80/20 splitting

In [8]:
split = int(len(df) * 0.8)

X_train = X.iloc[:split]        # iloc means "select by position number"
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

print(f"Training set:  {X_train.shape[0]} rows  ({X_train.index[0].date()} to {X_train.index[-1].date()})")
print(f"Test set:      {X_test.shape[0]} rows  ({X_test.index[0].date()} to {X_test.index[-1].date()})")

Training set:  1973 rows  (2015-03-16 to 2023-01-12)
Test set:      494 rows  (2023-01-13 to 2024-12-31)


fit_transform on training data — it learns the scaling from training data <br>
transform only on test data — it applies the same scaling without re-learning <br><br>

Never fit the scaler on test data. That would leak information from the future.

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)